In [0]:
print("Databricks Daily Sales Pipeline")

In [0]:
from pyspark.sql.functions import col, sum

input_file = "/Workspace/Users/dianawojdylak@gmail.com/data-engineering-lifecycle/databricks/daily_sales.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(input_file)
)

print(f"Rows read: {df.count()}")

df.show()

In [0]:
df = (
    df
    .withColumn("order_date", col("order_date").cast("date"))
    .filter(
        (col("quantity") > 0)
        & (col("unit_price") >= 0)
    )
    .withColumn(
        "revenue",
        col("quantity") * col("unit_price")
    )
)

df.show()

In [0]:
summary = (
    df.groupBy("order_date", "product")
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("revenue").alias("total_revenue"),
    )
    .orderBy("order_date", "product")
    )

summary.show()

In [0]:
summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.daily_sales_summary"
)

In [0]:
%sql
SELECT
    order_date,
    product,
    total_quantity,
    total_revenue
FROM workspace.default.daily_sales_summary
ORDER BY order_date, product;